# Задание лабораторной работы

На основе рассмотренного на лекции примера реализовать алгоритм Policy Iteration для любой среды обучения с подкреплением (кроме рассмотренной на лекции среды Toy Text / Frozen Lake) из библиотеки Gym (или аналогичной библиотеки).

# Выполнение работы

## Описание среды

Возьмём из библиотеки Gym среду Taxi-v3: https://www.gymlibrary.dev/environments/toy_text/taxi/

Задача представляет собой задачу о такси из книги Тома Диттериха "Обучение с иерархическим подкреплением с помощью декомпозиции функции MAXQ Value".

На карте есть 4 определенных места, обозначенных R(ed), G(reen), Y(ellow) и B(lue). Когда начинается поездка, такси выезжает из случайного квадрата, а пассажир оказывается в случайном месте. Такси подъезжает к месту нахождения пассажира, забирает его, отвозит в пункт назначения (другое из 4 указанных мест), а затем высаживает пассажира. Как только пассажир высажен, поездка заканчивается.

Есть 500 состояний:
- карта размером 5x5;
- 4 локации;
- 5 состояний пассажира (4 выхода и в такси).

Есть 6 действий:
- 0: двигаться на юг;
- 1: двигаться на север;
- 2: двигаться на запад;
- 3: двигаться на восток;
- 4: посадить пассажира;
- 5: высадить пассажира.

Существует 400 состояний, до которых можно добраться во время поездки. Пропущенные состояния соответствуют ситуациям, в которых пассажир находится в том же месте, что и пункт назначения, поскольку это обычно сигнализирует об окончании поездки. 4 дополнительных состояния можно наблюдать сразу после успешного завершения поездки, когда и пассажир, и такси находятся в пункте назначения. Всего получается 404 доступных дискретных состояния.

Каждое пространство состояний представлено кортежем: (taxi_row, taxi_col, passenger_location, destination).

Точки посадки пассажира:
- 0: R(ed);
- 1: G(reen);
- 2: Y(ellow);
- 3: B(lue);
- 4: в такси.

Пункты назначения (пункты высадки):
- 0: R(ed);
- 1: G(reen);
- 2: Y(ellow);
- 3: B(lue).

Награды:
- -1 за каждый шаг, если не предусмотрено иное вознаграждение;
- +20 за доставку пассажира;
- -10 за некорректное выполнение действий "погрузка" и "высадка".

In [38]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
import pandas as pd
from gymnasium.envs.toy_text.taxi import TaxiEnv


In [39]:
class PolicyIterationAgent:
    '''
    Класс, эмулирующий работу агента
    '''
    def __init__(self, env):
        self.env = env
        # Пространство состояний
        self.observation_dim = 500
        # Массив действий в соответствии с документацией
        self.actions_variants = np.array([0,1,2,3,4,5])
        # Задание стратегии (политики)
        self.policy_probs = np.full((self.observation_dim, len(self.actions_variants)), 0.16666666)
        # Начальные значения для v(s)
        self.state_values = np.zeros(shape=(self.observation_dim))
        # Начальные значения параметров
        self.maxNumberOfIterations = 1000
        self.theta=1e-6
        self.gamma=0.99


    def print_policy(self):
        '''
        Вывод матриц стратегии
        '''
        if self.policy_probs[0][0] != 0.16666666:
            x = TaxiEnv()
            pos = {0:'R', 1:'G',2:'Y', 3:'B', 4:'T'}
            print('''
            +---------+
            |R: | : :G|
            | : | : : |
            | : : : : |
            | | : | : |
            |Y| : |B: |
            +---------+
            ''')
            print('состояние: x,y,пассажир,назначение')
            print('Стратегия (первые 20 состояний):')
            for i in range(min(20, len(self.policy_probs))):
              t_x,t_y,passeng,dest = x.decode(i)
              action_names = ['Юг', 'Север', 'Запад', 'Восток', 'Взять', 'Высадить']
              best_action = np.argmax(self.policy_probs[i])
              print(f"({t_x},{t_y},{pos[passeng]},{pos[dest]}) -> {action_names[best_action]}")
        else:
            print('Стратегия: равномерная (не обучена)')


    def policy_evaluation(self):
        '''
        Оценивание стратегии
        '''
        valueFunctionVector = self.state_values.copy()
        for iterations in range(self.maxNumberOfIterations):
            valueFunctionVectorNextIteration=np.zeros(shape=(self.observation_dim))

            for state in range(self.observation_dim):
                action_probabilities = self.policy_probs[state]
                outerSum=0

                for action, prob in enumerate(action_probabilities):
                    innerSum=0
                    # В Taxi-v4 возвращается 4 элемента
                    transitions = self.env.unwrapped.P[state][action]
                    for transition in transitions:
                        probability, next_state, reward, terminated = transition
                        innerSum += probability * (reward + self.gamma * self.state_values[next_state])
                    outerSum += self.policy_probs[state][action] * innerSum

                valueFunctionVectorNextIteration[state] = outerSum

            if np.max(np.abs(valueFunctionVectorNextIteration - valueFunctionVector)) < self.theta:
                valueFunctionVector = valueFunctionVectorNextIteration
                print(f"  Оценивание сошлось за {iterations+1} итераций")
                break
            valueFunctionVector = valueFunctionVectorNextIteration

        return valueFunctionVector


    def policy_improvement(self):
        '''
        Улучшение стратегии
        '''
        qvaluesMatrix = np.zeros((self.observation_dim, len(self.actions_variants)))
        improvedPolicy = np.zeros((self.observation_dim, len(self.actions_variants)))

        for state in range(self.observation_dim):
            for action in range(len(self.actions_variants)):
                transitions = self.env.unwrapped.P[state][action]
                for transition in transitions:
                    probability, next_state, reward, terminated = transition
                    qvaluesMatrix[state, action] += probability * (reward + self.gamma * self.state_values[next_state])

            bestActionIndex = np.where(qvaluesMatrix[state,:] == np.max(qvaluesMatrix[state,:]))
            improvedPolicy[state, bestActionIndex] = 1 / np.size(bestActionIndex)

        return improvedPolicy


    def policy_iteration(self, cnt):
        '''
        Основная реализация алгоритма
        '''
        for i in range(1, cnt+1):
            print(f"Итерация {i}:")
            self.state_values = self.policy_evaluation()
            new_policy = self.policy_improvement()

            if np.array_equal(self.policy_probs, new_policy):
                print(f'Политика стабилизировалась на итерации {i}')
                self.policy_probs = new_policy
                break

            self.policy_probs = new_policy
        print(f'Алгоритм выполнился за {i} итераций.')


In [40]:
def play_agent(agent):
    env2 = gym.make('Taxi-v4', render_mode='human')
    state, _ = env2.reset()
    done = False
    total_reward = 0
    steps = 0

    while not done:
        p = agent.policy_probs[state]
        action = np.argmax(p)  # Выбираем лучшее действие

        next_state, reward, terminated, truncated, _ = env2.step(action)
        total_reward += reward
        steps += 1
        state = next_state

        if terminated or truncated:
            done = True

    print(f"\nЗавершено за {steps} шагов с наградой {total_reward}")
    env2.close()


In [41]:
# Обучение агента
env = gym.make('Taxi-v4')
env.reset()
agent = PolicyIterationAgent(env)

print("=" * 50)
print("ДО ОБУЧЕНИЯ:")
print("=" * 50)
agent.print_policy()

print("\n" + "=" * 50)
print("ПРОЦЕСС ОБУЧЕНИЯ:")
print("=" * 50)
agent.policy_iteration(40)

print("\n" + "=" * 50)
print("ПОСЛЕ ОБУЧЕНИЯ:")
print("=" * 50)
agent.print_policy()


ДО ОБУЧЕНИЯ:
Стратегия: равномерная (не обучена)

ПРОЦЕСС ОБУЧЕНИЯ:
Итерация 1:
  Оценивание сошлось за 2 итераций
Итерация 2:
  Оценивание сошлось за 2 итераций
Итерация 3:
  Оценивание сошлось за 2 итераций
Итерация 4:
  Оценивание сошлось за 2 итераций
Итерация 5:
  Оценивание сошлось за 2 итераций
Итерация 6:
  Оценивание сошлось за 2 итераций
Итерация 7:
  Оценивание сошлось за 2 итераций
Итерация 8:
  Оценивание сошлось за 2 итераций
Итерация 9:
  Оценивание сошлось за 2 итераций
Итерация 10:
  Оценивание сошлось за 2 итераций
Итерация 11:
  Оценивание сошлось за 2 итераций
Итерация 12:
  Оценивание сошлось за 2 итераций
Итерация 13:
  Оценивание сошлось за 2 итераций
Итерация 14:
  Оценивание сошлось за 2 итераций
Итерация 15:
  Оценивание сошлось за 2 итераций
Итерация 16:
  Оценивание сошлось за 2 итераций
Итерация 17:
  Оценивание сошлось за 2 итераций
Итерация 18:
  Оценивание сошлось за 2 итераций
Политика стабилизировалась на итерации 18
Алгоритм выполнился за 18 итераций.

In [42]:
# Проигрывание сцены
print("\n" + "=" * 50)
print("ДЕМОНСТРАЦИЯ РАБОТЫ АГЕНТА:")
print("=" * 50)
play_agent(agent)



ДЕМОНСТРАЦИЯ РАБОТЫ АГЕНТА:

Завершено за 12 шагов с наградой 9
